# 04 - Despliegue y Puesta en Producción

## 1. Descripción General

Una vez finalizado el proceso de exploración, preparación de datos, ingeniería de características, así como el entrenamiento, evaluación y selección final del modelo, el siguiente paso consiste en preparar el sistema para su utilización en un entorno de producción.

En este notebook se construye y valida el flujo de inferencia que posteriormente será utilizado por la aplicación. A diferencia del proceso de entrenamiento, donde el objetivo es optimizar y evaluar el modelo, aquí el enfoque se centra en garantizar que un nuevo listing pueda recorrer correctamente todo el pipeline de producción: desde la preparación de sus variables, la generación de la predicción y el cálculo de métricas complementarias, hasta la obtención de una respuesta lista para ser consumida por una interfaz de usuario.

Este notebook no implementa la aplicación final ni el servicio web. Su propósito es validar de manera aislada los distintos módulos de inferencia antes de integrarlos con FastAPI y Streamlit en la etapa de despliegue.

En particular, su alcance incluye:

- Verificar la carga correcta de los artefactos de producción.
- Validar el pipeline completo de preparación de características para nuevos listings.
- Generar predicciones utilizando el modelo entrenado.
- Calcular métricas complementarias de mercado, como el precio típico, el rango de precios y la posición relativa del listing frente a alojamientos comparables.
- Comprobar el funcionamiento de los módulos que serán reutilizados durante el despliegue de la aplicación.

Al finalizar este notebook, se contará con un pipeline de inferencia completamente funcional y validado, listo para ser integrado en la API y la interfaz interactiva que conformarán la aplicación final.

## 2. Importación de Librerías y Carga del Dataset de Prueba

In [1]:
# Import libraries and modules
import pandas as pd
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

# Load the autoreload extension to automatically reload modules when they change
# Set autoreload mode to 2: reload all modules before executing user code
%load_ext autoreload
%autoreload 2

# Import service modules for feature preparation, inference, and insights
from app.services.features import prepare_features
from app.services.inference import predict_listing
from app.services.insights import generate_market_insights

# Import feature list from model configuration
from src.settings.model_settings import MODEL_FEATURES

# Load the dataset generated before the Feature Engineering phase, to be used for testing the deployment pipeline.
df_test = pd.read_csv("../data/processed/df_features.csv")

# Preview first rows
df_test.head()

,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,minimum_nights,...,instant_bookable,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,clean_price,log_price
0,Cuajimalpa de Morelos,19.38283,-99.27178,Entire villa,Entire home/apt,2,1.0,1.0,1.0,1,...,f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3673.0,8.208764
1,Cuauhtémoc,19.41162,-99.17794,Entire home,Entire home/apt,14,5.5,5.0,8.0,1,...,f,4.59,4.56,4.70,4.87,4.78,4.98,4.47,18000.0,9.798127
2,Cuauhtémoc,19.43977,-99.15605,Entire condo,Entire home/apt,2,1.0,1.0,1.0,15,...,f,4.87,4.95,4.88,4.98,4.94,4.76,4.79,591.0,6.381816
3,Miguel Hidalgo,19.40826,-99.18659,Entire home,Entire home/apt,16,5.0,5.0,10.0,2,...,f,4.88,4.91,4.84,4.91,4.89,4.75,4.90,3673.0,8.208764
4,Benito Juárez,19.39675,-99.17581,Private room in rental unit,Private room,2,1.0,1.0,1.0,4,...,f,4.84,4.86,4.61,4.98,4.95,4.97,4.83,321.0,5.771441


## 3. Prueba del Pipeline de Preprocesamiento e Inferencia

Antes de integrar el modelo en una API y una interfaz de usuario, es importante verificar que el pipeline de producción funcione correctamente de principio a fin. En esta sección se realiza una prueba utilizando un listing del conjunto de prueba, simulando el flujo que seguirá un nuevo alojamiento al utilizar la aplicación.

Durante la inferencia, el listing atraviesa todas las etapas necesarias para generar una recomendación de precio: preparación de características, aplicación del pipeline de preprocesamiento, predicción mediante el modelo entrenado y cálculo de métricas complementarias de mercado. El objetivo no es volver a evaluar el desempeño del modelo, sino validar que todos los componentes de producción interactúan correctamente y producen una salida coherente.

In [2]:
# END-TO-END PREPROCESSING & INFERENCE TEST

# Select one listing from the test set
listing = df_test.iloc[[0]].copy()

print()
print("RAW LISTING:")
print(f"Shape: {listing.shape}")
display(listing.head())

# Prepare features
prepared_listing = prepare_features(listing)

print()
print("PREPARED FEATURES:")
print(f"Shape: {prepared_listing.shape}")
display(prepared_listing.head())

# Generate prediction
prediction = predict_listing(listing)

# Generate market insights
insights = generate_market_insights(
    listing=listing,
    predicted_price=prediction
)

# Display results
print()
print("=" * 60)
print("SMART PRICING RESULTS")
print("=" * 60)

print(f"Estimated Price      : ${prediction:,.2f} MXN")
print(f"Typical Market Price : ${insights['market_median']:,.2f} MXN")
print(
    f"Market Price Range   : "
    f"${insights['market_price_min']:,.2f}"
    f" - "
    f"${insights['market_price_max']:,.2f} MXN"
)
print(f"Listing Position     : {insights['listing_position']}")
print(f"Comparables listings : {insights['num_comparables']}")
print(f"Confidence           : {insights['confidence']}")
print(f"Search Radius (Km)   : {insights['search_radius_km']}")


RAW LISTING:
Shape: (1, 26)


,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,minimum_nights,...,instant_bookable,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,clean_price,log_price
0,Cuajimalpa de Morelos,19.38283,-99.27178,Entire villa,Entire home/apt,2,1.0,1.0,1.0,1,...,f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3673.0,8.208764



PREPARED FEATURES:
Shape: (1, 27)


,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,dist_to_nearest_attraction,beds,amenity_score,accommodates,bedrooms,bathrooms,attractions_within_radius,commercial_within_radius,...,host_is_superhost,instant_bookable,has_tv,has_elevator,has_free_parking,has_coffee_maker,has_outdoor_furniture,has_air_conditioning,has_self_check_in,has_pool
0,1,0,2.45099,1.0,0.26838,2,1.0,1.0,0,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0



SMART PRICING RESULTS
Estimated Price      : $1,180.08 MXN
Typical Market Price : $1,205.50 MXN
Market Price Range   : $948.25 - $1,622.75 MXN
Listing Position     : Fairly Priced
Comparables listings : 90
Confidence           : High
Search Radius (Km)   : 2.0


La prueba confirma que el pipeline de inferencia funciona correctamente de extremo a extremo. A partir de un nuevo listing, el sistema aplica automáticamente la preparación de características, el pipeline de preprocesamiento entrenado y el modelo de regresión (XGBoost) para generar una recomendación de precio, sin requerir intervención manual.

Para este alojamiento, el modelo estima un precio de \$1,180.08 MXN, valor que se encuentra dentro del rango de precios observado en el mercado para alojamientos comparables (\$948.25 – \$1,622.75 MXN). Asimismo, el precio típico de estos comparables es de \$1,205.50 MXN, lo que indica que la recomendación del modelo es consistente con el comportamiento del mercado local.

La búsqueda identificó 90 alojamientos comparables utilizando un radio de 2 km, proporcionando una base suficiente para calcular las métricas complementarias. Debido al número de comparables encontrados, el sistema asignó un nivel de confianza High, sugiriendo que la estimación se apoya en una muestra representativa de listings similares. En conjunto, estos resultados validan que el pipeline de producción no solo genera una predicción del precio, sino también información contextual que facilita la interpretación de dicha recomendación.


Con esta validación concluye el desarrollo del pipeline de inferencia del proyecto. Se comprobó que un nuevo listing puede recorrer correctamente todas las etapas necesarias para generar una recomendación de precio, desde la preparación de características y el preprocesamiento hasta la predicción del modelo y el cálculo de métricas complementarias de mercado. Con el pipeline de inferencia validado, el siguiente paso consiste en integrarlo dentro de una aplicación interactiva que permita a cualquier usuario obtener recomendaciones de precio sin necesidad de ejecutar código.

A partir de este punto, el proyecto está listo para ser integrado en una aplicación interactiva. La implementación del sistema se realizará utilizando **FastAPI** como capa de servicio para exponer el modelo mediante una API REST y **Streamlit** como interfaz de usuario para capturar las características de un alojamiento y presentar las recomendaciones de precio junto con los insights de mercado.


Las pruebas realizadas en este notebook concluyen la validación del pipeline de producción y representan la última etapa del desarrollo del modelo dentro de este proyecto. A partir de este punto, el trabajo continuará directamente en la implementación de la aplicación, donde el pipeline será integrado con FastAPI y Streamlit. Las validaciones restantes corresponderán al correcto funcionamiento de la API, la comunicación con la interfaz y la experiencia de usuario, sin requerir nuevas evaluaciones del modelo predictivo o nuevas validaciones en este notebook.

In [7]:
pd.set_option("display.max_columns", None)
listing

,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,minimum_nights,amenities,host_is_superhost,host_verifications,host_total_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,instant_bookable,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,clean_price,log_price
0,Cuajimalpa de Morelos,19.38283,-99.27178,Entire villa,Entire home/apt,2,1.0,1.0,1.0,1,"[""Garden view"", ""Resort access"", ""Washer"", ""Co...",f,"['email', 'phone', 'work_email']",1.0,1,0,f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3673.0,8.208764


In [8]:
prepared_listing

,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,dist_to_nearest_attraction,beds,amenity_score,accommodates,bedrooms,bathrooms,attractions_within_radius,commercial_within_radius,room_type,neighbourhood_cleansed,property_group_room,host_total_listings_segment,review_scores_mean_segment,minimum_nights_segment,host_verifications_grouped,host_is_superhost,instant_bookable,has_tv,has_elevator,has_free_parking,has_coffee_maker,has_outdoor_furniture,has_air_conditioning,has_self_check_in,has_pool
0,1,0,2.45099,1.0,0.26838,2,1.0,1.0,0,1,entire_home/apt,Cuajimalpa de Morelos,house_entire_home/apt,small_host,NaN,short_stay,extended,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [6]:
print(listing['amenities'].iloc[0])

["Garden view", "Resort access", "Washer", "Courtyard view", "Kitchen", "BBQ grill", "Free parking on premises", "Bed linens", "Wifi", "Hot water", "Pocket wifi", "Indoor fireplace"]
